In [1]:
import ee
ee.Authenticate()

Enter verification code:  4/1ATsMZqAckIQ7HoTD0EmXTybMsL4f-lt0JatjnShpUW0-_126amj4EE4-2xY



Successfully saved authorization token.


In [2]:
ee.Initialize(project='rinde-soja')
print("Earth Engine listo!")

Earth Engine listo!


In [3]:
import ee
import pandas as pd
from pathlib import Path

ee.Initialize(project='rinde-soja')

# ============================================================
# 1. OBTENER POLÍGONOS DE LOS PARTIDOS DESDE FAO GAUL (está en GEE)
# ============================================================
admin2 = ee.FeatureCollection("FAO/GAUL/2015/level2")
buenos_aires = admin2.filter(ee.Filter.eq("ADM1_NAME", "Buenos Aires"))

# Verificar nombres disponibles
nombres_ba = buenos_aires.aggregate_array("ADM2_NAME").getInfo()

# Mapeo de nuestros partidos a nombres en FAO GAUL (pueden variar levemente)
PARTIDOS_BUSCAR = ["General Arenales", "Leandro N. Alem", "Junín", "Lincoln", "General Pinto"]

print("Buscando partidos en la base de datos...")
partidos_encontrados = {}
for p in PARTIDOS_BUSCAR:
    # Buscar coincidencia exacta o parcial
    matches = [n for n in nombres_ba if p.lower() in n.lower()]
    if matches:
        partidos_encontrados[p] = matches[0]
        print(f"  ✓ {p} → '{matches[0]}'")
    else:
        print(f"  ✗ {p} → NO ENCONTRADO")

# Filtrar solo los partidos que encontramos
partidos_fc = buenos_aires.filter(
    ee.Filter.inList("ADM2_NAME", list(partidos_encontrados.values()))
)
print(f"\nPartidos cargados: {partidos_fc.size().getInfo()}")

# ============================================================
# 2. EXTRAER NDVI DE MODIS (MOD13Q1) POR PARTIDO Y POR FECHA
# ============================================================
# MOD13Q1: NDVI cada 16 días, resolución 250m, desde 2000
modis = ee.ImageCollection("MODIS/061/MOD13Q1") \
    .filterDate("2000-01-01", "2025-04-30") \
    .select("NDVI")

print(f"Imágenes MODIS disponibles: {modis.size().getInfo()}")
print("Extrayendo NDVI por partido (esto tarda 5-10 minutos)...")

# Extraer año por año para no saturar la memoria de GEE
all_results = []

for year in range(2000, 2026):
    # Filtrar campaña completa (sep del año anterior a mayo)
    start = f"{year}-01-01"
    end = f"{year}-12-31"
    if year == 2025:
        end = "2025-04-30"
    
    yearly = modis.filterDate(start, end)
    n_images = yearly.size().getInfo()
    if n_images == 0:
        continue
    
    # Para cada imagen, calcular NDVI promedio por partido
    def extract_ndvi(image):
        date = image.date().format("YYYY-MM-dd")
        means = image.reduceRegions(
            collection=partidos_fc,
            reducer=ee.Reducer.mean(),
            scale=250
        )
        return means.map(lambda f: f.set("fecha", date))
    
    results = yearly.map(extract_ndvi).flatten()
    
    # Traer a Python
    try:
        data = results.getInfo()
        for feat in data["features"]:
            props = feat["properties"]
            all_results.append({
                "fecha": props.get("fecha"),
                "partido_gaul": props.get("ADM2_NAME"),
                "ndvi_raw": props.get("mean"),
            })
        print(f"  {year}: {n_images} imágenes — OK")
    except Exception as e:
        print(f"  {year}: ERROR — {e}")

# ============================================================
# 3. ARMAR DATAFRAME Y GUARDAR
# ============================================================
df_ndvi = pd.DataFrame(all_results)

# MODIS guarda NDVI con factor de escala 0.0001
df_ndvi["ndvi"] = df_ndvi["ndvi_raw"] * 0.0001

# Mapear nombres de vuelta
nombre_inverso = {v: k for k, v in partidos_encontrados.items()}
df_ndvi["partido"] = df_ndvi["partido_gaul"].map(nombre_inverso)

df_ndvi["fecha"] = pd.to_datetime(df_ndvi["fecha"])
df_ndvi = df_ndvi.sort_values(["partido", "fecha"]).reset_index(drop=True)

# Guardar — usá tu ruta local
output_path = Path.home() / "Documents" / "rinde-soja" / "data" / "raw" / "ndvi" / "ndvi_modis_zona_coop_2000_2025.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df_ndvi.to_csv(output_path, index=False)

print(f"\n{'='*50}")
print(f"Guardado en: {output_path}")
print(f"Shape: {df_ndvi.shape}")
print(f"Período: {df_ndvi['fecha'].min().date()} a {df_ndvi['fecha'].max().date()}")
print(f"Partidos: {df_ndvi['partido'].unique().tolist()}")
print(f"NDVI rango: {df_ndvi['ndvi'].min():.3f} a {df_ndvi['ndvi'].max():.3f}")

Buscando partidos en la base de datos...
  ✓ General Arenales → 'General Arenales'
  ✓ Leandro N. Alem → 'Leandro N. Alem'
  ✗ Junín → NO ENCONTRADO
  ✓ Lincoln → 'Lincoln'
  ✓ General Pinto → 'General Pinto'

Partidos cargados: 4
Imágenes MODIS disponibles: 580
Extrayendo NDVI por partido (esto tarda 5-10 minutos)...
  2000: 20 imágenes — OK
  2001: 23 imágenes — OK
  2002: 23 imágenes — OK
  2003: 23 imágenes — OK
  2004: 23 imágenes — OK
  2005: 23 imágenes — OK
  2006: 23 imágenes — OK
  2007: 23 imágenes — OK
  2008: 23 imágenes — OK
  2009: 23 imágenes — OK
  2010: 23 imágenes — OK
  2011: 23 imágenes — OK
  2012: 23 imágenes — OK
  2013: 23 imágenes — OK
  2014: 23 imágenes — OK
  2015: 23 imágenes — OK
  2016: 23 imágenes — OK
  2017: 23 imágenes — OK
  2018: 23 imágenes — OK
  2019: 23 imágenes — OK
  2020: 23 imágenes — OK
  2021: 23 imágenes — OK
  2022: 23 imágenes — OK
  2023: 23 imágenes — OK
  2024: 23 imágenes — OK
  2025: 8 imágenes — OK

Guardado en: C:\Users\agust\Do

In [4]:
# Buscar cómo se llama Junín en la base FAO GAUL
admin2 = ee.FeatureCollection("FAO/GAUL/2015/level2")
buenos_aires = admin2.filter(ee.Filter.eq("ADM1_NAME", "Buenos Aires"))
nombres = buenos_aires.aggregate_array("ADM2_NAME").getInfo()

# Buscar variantes de "Jun"
junin_matches = [n for n in nombres if "jun" in n.lower()]
print("Coincidencias encontradas:", junin_matches)

Coincidencias encontradas: ['Junin']


In [5]:
import ee
import pandas as pd
from pathlib import Path

ee.Initialize(project='rinde-soja')

admin2 = ee.FeatureCollection("FAO/GAUL/2015/level2")
junin_fc = admin2.filter(ee.Filter.eq("ADM2_NAME", "Junin"))

modis = ee.ImageCollection("MODIS/061/MOD13Q1") \
    .filterDate("2000-01-01", "2025-04-30") \
    .select("NDVI")

print("Descargando NDVI para Junín...")
all_results = []

for year in range(2000, 2026):
    start = f"{year}-01-01"
    end = f"{year}-12-31" if year < 2025 else "2025-04-30"
    
    yearly = modis.filterDate(start, end)
    n_images = yearly.size().getInfo()
    if n_images == 0:
        continue
    
    def extract_ndvi(image):
        date = image.date().format("YYYY-MM-dd")
        means = image.reduceRegions(
            collection=junin_fc,
            reducer=ee.Reducer.mean(),
            scale=250
        )
        return means.map(lambda f: f.set("fecha", date))
    
    results = yearly.map(extract_ndvi).flatten()
    
    try:
        data = results.getInfo()
        for feat in data["features"]:
            props = feat["properties"]
            all_results.append({
                "fecha": props.get("fecha"),
                "partido_gaul": "Junin",
                "ndvi_raw": props.get("mean"),
            })
        print(f"  {year}: OK")
    except Exception as e:
        print(f"  {year}: ERROR — {e}")

# Armar DataFrame de Junín
df_junin = pd.DataFrame(all_results)
df_junin["ndvi"] = df_junin["ndvi_raw"] * 0.0001
df_junin["partido"] = "Junín"
df_junin["fecha"] = pd.to_datetime(df_junin["fecha"])

# Leer CSV existente y agregar Junín
csv_path = Path.home() / "Documents" / "rinde-soja" / "data" / "raw" / "ndvi" / "ndvi_modis_zona_coop_2000_2025.csv"
df_existente = pd.read_csv(csv_path, parse_dates=["fecha"])
df_final = pd.concat([df_existente, df_junin], ignore_index=True)
df_final = df_final.sort_values(["partido", "fecha"]).reset_index(drop=True)

# Sobreescribir el CSV con los 5 partidos
df_final.to_csv(csv_path, index=False)

print(f"\nListo! Shape final: {df_final.shape}")
print(f"Partidos: {df_final['partido'].unique().tolist()}")

Descargando NDVI para Junín...
  2000: OK
  2001: OK
  2002: OK
  2003: OK
  2004: OK
  2005: OK
  2006: OK
  2007: OK
  2008: OK
  2009: OK
  2010: OK
  2011: OK
  2012: OK
  2013: OK
  2014: OK
  2015: OK
  2016: OK
  2017: OK
  2018: OK
  2019: OK
  2020: OK
  2021: OK
  2022: OK
  2023: OK
  2024: OK
  2025: OK

Listo! Shape final: (5800, 5)
Partidos: ['General Arenales', 'General Pinto', 'Junín', 'Leandro N. Alem', 'Lincoln']
